# ICBHI RespFusionNet — Modality Ablation Study

This notebook runs a *modality* ablation on the ICBHI 2017 Respiratory Sound
Database using `RespFusionNet` — a minimal multimodal fusion classifier
inspired by the RespLLM hypothesis that combining respiratory audio with
patient context improves prediction over unimodal inputs. RespFusionNet is
deliberately *not* a reproduction of RespLLM (no OpenBioLLM, no OPERA, no
LoRA) — it is a small, transparent baseline that exposes the audio vs.
metadata contrast at the PyHealth-model level.

The companion notebook `icbhi_respiratory_classification.ipynb` ablates over
**label modes** (`any_abnormal` / `crackle_only` / `wheeze_only`) with a
unimodal audio CNN — it is the safer dataset+task submission path and is
unaffected by this notebook.

What this notebook demonstrates:

1. Loading the ICBHI dataset via `ICBHIDataset`.
2. Applying `RespiratoryAbnormalityPredictionICBHI` with
   `include_metadata_features=True` — which surfaces a fixed 7-dim patient/
   context vector alongside the existing `signal` tensor.
3. **Ablation over three modality configurations** with `RespFusionNet`:
   audio only, metadata only, audio + metadata.
4. Reporting per-configuration class balance and test accuracy.

**Running this notebook:**

- Set `DEMO = True` (default) to run on purely synthetic data — no download.
- Set `DEMO = False` and point `ROOT` at your `ICBHI_final_database/`
  directory for real results.

In demo mode the audio is zero-filled, so only the metadata branch carries
any signal — this is intentional, it confirms the fusion plumbing runs
end-to-end but says nothing scientifically.

In [ ]:
# ---------------------------------------------------------------------------
# Configuration — adjust before running
# ---------------------------------------------------------------------------
ROOT = "/data/ICBHI_final_database"   # used only when DEMO=False
CACHE_DIR = ".cache/icbhi_respfusionnet"

DEMO = True           # True -> synthetic data, no download needed

RESAMPLE_RATE = 4000  # Hz — low enough to capture respiratory acoustics
TARGET_LENGTH = 2.0   # seconds — pad/trim each cycle to this fixed length
LABEL_MODE = "any_abnormal"  # most balanced mode — see sibling notebook

HIDDEN_DIM = 64       # RespFusionNet per-branch MLP hidden size
DROPOUT = 0.1
BATCH_SIZE = 4
EPOCHS = 3            # short for demo; use 20+ with real data
LR = 1e-3

In [ ]:
import shutil
import tempfile
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from pyhealth.datasets import ICBHIDataset
from pyhealth.models import RespFusionNet
from pyhealth.tasks import RespiratoryAbnormalityPredictionICBHI

## Demo Fixture

When `DEMO=True` we synthesize a minimal ICBHI-shaped directory: 5 patients,
1 recording each, 2 cycles per recording, covering all four crackle/wheeze
annotation combinations. Audio is zero-filled (no real acoustic content);
training will not converge, but every code path executes so the ablation
output structure is faithful.

This fixture duplicates the helper from the sibling notebook intentionally
— the two notebooks are independent files and do not share imports.

In [ ]:
def _build_demo_root(root: Path) -> None:
    """Populate *root* with a minimal ICBHI-shaped synthetic dataset."""
    import scipy.io.wavfile

    sample_rate = RESAMPLE_RATE
    audio = np.zeros(sample_rate * 4, dtype=np.int16)  # 4 s silent clip

    # (pid, rec, location, mode, equip, split, [(start,end,crackle,wheeze)...])
    recordings = [
        ("101", "1b1", "Ar", "sc", "Meditron", "train",
         [(0.0, 1.5, 0, 0), (1.5, 3.0, 1, 0)]),
        ("102", "1b1", "Pl", "mc", "AKGC417L", "train",
         [(0.0, 1.5, 0, 1), (1.5, 3.0, 1, 1)]),
        ("103", "1b1", "Ar", "sc", "Meditron", "train",
         [(0.0, 1.5, 0, 0), (1.5, 3.0, 0, 1)]),
        ("104", "1b1", "Ar", "sc", "Meditron", "test",
         [(0.0, 1.5, 1, 0), (1.5, 3.0, 1, 1)]),
        ("105", "1b1", "Ar", "sc", "Meditron", "test",
         [(0.0, 1.5, 0, 0), (1.5, 3.0, 0, 1)]),
    ]

    train_stems, test_stems = [], []
    for pid, rec, loc, mode, equip, split, anns in recordings:
        stem = f"{pid}_{rec}_{loc}_{mode}_{equip}"
        scipy.io.wavfile.write(str(root / f"{stem}.wav"), sample_rate, audio)
        ann_text = "\n".join(f"{s}\t{e}\t{c}\t{w}" for s, e, c, w in anns)
        (root / f"{stem}.txt").write_text(ann_text + "\n")
        (train_stems if split == "train" else test_stems).append(stem)

    (root / "ICBHI_challenge_diagnosis.txt").write_text(
        "101 Healthy\n102 URTI\n103 COPD\n104 URTI\n105 Healthy\n"
    )
    (root / "ICBHI_Challenge_demographic_information.txt").write_text(
        "101 45 M 22.5 NA NA\n102 5 F NA 18.2 108.0\n"
        "103 67 M 28.1 NA NA\n104 12 F NA 21.0 140.0\n105 55 M 24.3 NA NA\n"
    )
    split_dir = root / "ICBHI_challenge_train_and_test_txt"
    split_dir.mkdir()
    (split_dir / "ICBHI_challenge_train.txt").write_text(
        "\n".join(train_stems) + "\n"
    )
    (split_dir / "ICBHI_challenge_test.txt").write_text(
        "\n".join(test_stems) + "\n"
    )

## Collate + Ablation Runner

`collate_modality` stacks only the tensors RespFusionNet needs from each
sample dict: `signal`, `metadata`, and `label`. Extra keys like `split`,
`patient_id`, etc. are silently dropped — RespFusionNet consumes `**kwargs`
and indexes only the configured feature / label keys.

`run_modality` trains and evaluates one `RespFusionNet` configuration
(audio-only, metadata-only, or both) on the same shared task samples.

In [ ]:
def collate_modality(batch):
    """Stack signal, metadata, label across a batch of sample dicts."""
    return {
        "signal": torch.stack([s["signal"] for s in batch]),
        "metadata": torch.stack([s["metadata"] for s in batch]),
        "label": torch.stack([
            s["label"] if torch.is_tensor(s["label"])
            else torch.tensor([float(s["label"])])
            for s in batch
        ]),
    }


def run_modality(
    samples_dataset,
    train_samples,
    test_samples,
    use_audio: bool,
    use_metadata: bool,
) -> dict:
    """Train and evaluate RespFusionNet with one modality configuration.

    Args:
        samples_dataset: The SampleDataset returned by ``dataset.set_task(...)``.
            Used only by RespFusionNet's constructor for schema resolution
            (input_schema, output_processor) — training data comes from
            ``train_samples`` / ``test_samples``.
        train_samples: List of per-cycle sample dicts in the train split.
        test_samples: List of per-cycle sample dicts in the test split.
        use_audio: Enable the audio branch.
        use_metadata: Enable the metadata branch.

    Returns:
        Dict with configuration and metrics.
    """
    tag = []
    if use_audio:
        tag.append("audio")
    if use_metadata:
        tag.append("metadata")
    tag_str = " + ".join(tag) if tag else "(none)"

    print(f"\n{'='*60}")
    print(f"  ABLATION: modality = {tag_str}")
    print(f"{'='*60}")

    n_train, n_test = len(train_samples), len(test_samples)
    train_pos = sum(1 for s in train_samples if int(torch.as_tensor(
        s["label"]).item()) == 1)
    train_pos_pct = train_pos / n_train if n_train else 0.0
    print(
        f"  Train: {n_train} cycles  "
        f"({train_pos}/{n_train} positive = {train_pos_pct:.1%})"
    )
    print(f"  Test:  {n_test} cycles")

    if n_train == 0:
        print("  [SKIP] no training samples — returning NaN accuracy")
        return {
            "modality": tag_str, "n_train": 0, "n_test": n_test,
            "train_pos_pct": 0.0, "test_acc": float("nan"),
        }

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = RespFusionNet(
        dataset=samples_dataset,
        hidden_dim=HIDDEN_DIM,
        dropout=DROPOUT,
        use_audio=use_audio,
        use_metadata=use_metadata,
    ).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    train_loader = DataLoader(
        train_samples, batch_size=BATCH_SIZE, shuffle=True,
        collate_fn=collate_modality, num_workers=0,
    )

    for epoch in range(1, EPOCHS + 1):
        model.train()
        total_loss, seen = 0.0, 0
        for batch in train_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            optimizer.zero_grad()
            out = model(**batch)
            out["loss"].backward()
            optimizer.step()
            bs = batch["label"].shape[0]
            total_loss += out["loss"].item() * bs
            seen += bs
        print(f"  Epoch {epoch}/{EPOCHS} — loss: {total_loss / seen:.4f}")

    if test_samples:
        test_loader = DataLoader(
            test_samples, batch_size=BATCH_SIZE, shuffle=False,
            collate_fn=collate_modality, num_workers=0,
        )
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for batch in test_loader:
                batch = {k: v.to(device) for k, v in batch.items()}
                out = model(**batch)
                preds = (out["y_prob"] >= 0.5).long().flatten()
                labels = batch["label"].long().flatten()
                correct += (preds == labels).sum().item()
                total += labels.numel()
        test_acc = correct / total if total else float("nan")
    else:
        test_acc = float("nan")

    print(f"  Test accuracy: {test_acc:.3f}")
    return {
        "modality": tag_str,
        "use_audio": use_audio,
        "use_metadata": use_metadata,
        "n_train": n_train,
        "n_test": n_test,
        "train_pos_pct": train_pos_pct,
        "test_acc": test_acc,
    }

## Load Dataset + Task

`include_metadata_features=True` is the single knob that makes the task
emit the 7-dim `metadata` tensor alongside `signal`. With the default
`False`, the task's behavior is unchanged — the sibling label-mode
notebook still runs byte-for-byte identically.

In [ ]:
_tmpdir = None

if DEMO:
    _tmpdir = tempfile.mkdtemp(prefix="icbhi_respfusion_")
    _demo_root = Path(_tmpdir)
    print(f"Demo mode: building synthetic fixture at {_demo_root}")
    _build_demo_root(_demo_root)
    _root = str(_demo_root)
    _cache = str(_demo_root / ".cache")
else:
    _root = ROOT
    _cache = CACHE_DIR

print(f"Loading ICBHIDataset from: {_root}")
dataset = ICBHIDataset(root=_root, subset="both", cache_dir=_cache)
dataset.stats()

task = RespiratoryAbnormalityPredictionICBHI(
    label_mode=LABEL_MODE,
    resample_rate=RESAMPLE_RATE,
    target_length=TARGET_LENGTH,
    include_metadata_features=True,
)
samples_dataset = dataset.set_task(task)

# Partition by the official ICBHI train/test annotation embedded in each
# sample. Iterating the SampleDataset yields dicts with "signal",
# "metadata", "label", and pass-through fields like "split".
all_samples = list(samples_dataset)
train_samples = [s for s in all_samples if s.get("split") == "train"]
test_samples = [s for s in all_samples if s.get("split") == "test"]
print(
    f"Total cycles: {len(all_samples)} "
    f"(train={len(train_samples)}, test={len(test_samples)})"
)

## Ablation: three modality configurations

| Config         | Audio branch | Metadata branch |
|----------------|:------------:|:---------------:|
| audio only     | ✓            | —               |
| metadata only  | —            | ✓               |
| audio + meta   | ✓            | ✓               |

All three configurations train a fresh `RespFusionNet` from scratch on the
same train/test samples. Only the enabled modality branches differ — the
classifier head, optimizer, and number of epochs are held constant.

**Expected observations on real ICBHI data (920 recordings, 6898 cycles):**

- *audio only* — learns adventitious-sound acoustic features; best of the
  unimodal baselines.
- *metadata only* — learns only from age/sex/BMI/duration; acts as a
  sanity-check lower bound. Useful for confirming that the audio branch is
  doing real work.
- *audio + metadata* — fusion; RespLLM's core hypothesis predicts that
  this configuration should beat both unimodal baselines.

*In demo mode the audio is zero-filled, so the audio-only branch has no
learnable signal — results here are structural ("the plumbing works"),
not scientific. Flip `DEMO=False` with real ICBHI data for a real test.*

In [ ]:
MODALITY_CONFIGS = [
    ("audio only",     True,  False),
    ("metadata only",  False, True),
    ("audio + metadata", True, True),
]

results = []
for _, use_a, use_m in MODALITY_CONFIGS:
    results.append(run_modality(
        samples_dataset=samples_dataset,
        train_samples=train_samples,
        test_samples=test_samples,
        use_audio=use_a,
        use_metadata=use_m,
    ))

In [ ]:
# ---------------------------------------------------------------------------
# Ablation summary table
# ---------------------------------------------------------------------------
print("\n" + "=" * 64)
print("  MODALITY ABLATION SUMMARY")
print("=" * 64)
print(f"  {'Config':<18} {'N train':>8} {'Train +%':>10} {'Test Acc':>10}")
print("  " + "-" * 54)
for r in results:
    acc = r["test_acc"]
    acc_str = f"{acc:.3f}" if acc == acc else "  N/A  "
    print(
        f"  {r['modality']:<18} {r['n_train']:>8} "
        f"{r['train_pos_pct']:>9.1%} {acc_str:>10}"
    )
print("=" * 64)
print(
    "\nInterpretation (real data):\n"
    "  audio only      — acoustic-feature baseline.\n"
    "  metadata only   — demographic baseline; lower bound.\n"
    "  audio + meta    — RespLLM-style fusion; expected to improve\n"
    "                    over both unimodal baselines if the metadata\n"
    "                    fields carry complementary information.\n"
)

In [ ]:
# Cleanup synthetic temp directory (no-op when DEMO=False)
if _tmpdir:
    shutil.rmtree(_tmpdir, ignore_errors=True)
    print(f"Cleaned up demo temp dir: {_tmpdir}")